# H1 mechanism: which of A's paragraphs must B see?

Frozen design from `JOURNAL.md`, “Direction: which attention does separate encoding lose?”. Run only after the H1 gap is established and `tests/test_masking.py` passes. Each condition recomputes the same B1 tokens at the same positions, with a causal mask. Only attention from B's paragraph tokens to A changes; the answer tail can attend to all tokens.

- `blocked`: B sees none of A; must reproduce B1-mask.
- `gold`: B sees all A gold paragraphs.
- `distractor`: B sees one deterministically sampled A distractor. This is undefined when A has none, so those rows are excluded from this arm and paired comparisons report the remaining sample.
- `except_gold`: B sees all of A except gold paragraph tokens.
- `full`: B sees all of A; must reproduce B1.

**Prediction:** gold access recovers more of the blocked-to-full gap than a distractor. Read AB and BA separately; AA is a placebo and BB is the colocated control; neither is assumed to have a zero gap. Compare gold vs distractor on paired rows and report counts and uncertainty. Paragraphs can differ in token length; the distractor control does not match a token budget. The interventions use gold labels and are not deployable selectors.

**Outcome:** exact match and teacher-forced log-probability of the first gold alias in `\boxed{...}`, conditional on the normal thinking-off answer tail. Sum and mean per token are saved. This is one particular answer string, not probability summed across aliases; compare the same question across conditions. Full recomputation is an upper bound on a partial-cache repair, not a latency measurement.

Full runs require Part 0 composition labels for the requested seeds. Smoke mode uses the first two unfiltered questions per cell so the runner audits a fixed sample; only full runs apply the composition filter. Smoke rows outside that subset do not enter full-run tables. Mask tensors use quadratic memory in context length.

In [ ]:
MODEL = "Qwen/Qwen3-4B"
DTYPE = "bfloat16"
SMOKE = True
SEEDS = [0, 1, 2]
N_PER_CELL = None
SKIP = []
DISTRACTOR_SEED = 0
MAX_ANSWER_TOKENS = 256

In [ ]:
import json
import pandas as pd
from latent_dispatch import analysis, arms, data, results
from latent_dispatch.model import Model
from latent_dispatch.results import load_results, results_path, run

## Questions and Part 0 filter

The toy uses `hop1` (two facts); MuSiQue uses two-hop questions. `AA`, `BB`, `AB`, and `BA` are kept separately. For full runs, missing or smoke-only Part 0 labels stop the run; no unlabeled question is silently treated as requiring composition.

In [ ]:
seeds = SEEDS[:1] if SMOKE else SEEDS
qs = (data.load("toy", seed=seeds, qtype="hop1", placement=["AA", "BB", "AB", "BA"])
      + data.load("musique", seed=seeds, qtype="2hop", placement=["AA", "BB", "AB", "BA"]))
qs = data.per_cell(qs, 2 if SMOKE else N_PER_CELL)
labels_path = results_path(MODEL, DTYPE, "x").parent / "part0_requires_composition.json"
if SMOKE:
    print("SMOKE ONLY: fixed unfiltered questions for the runner's output audit; not a result")
else:
    if not labels_path.exists():
        raise FileNotFoundError(f"Run Part 0 first or load a completed archive: {labels_path}")
    labels_file = json.loads(labels_path.read_text())
    if set(labels_file["seeds"]) != set(seeds) or labels_file["smoke"]:
        raise ValueError("Full runs require non-smoke Part 0 labels for exactly the requested seeds")
    labels = labels_file["labels"]
    missing = {f"{q['dataset']}:{q['source_id']}" for q in qs} - labels.keys()
    if missing:
        raise ValueError(f"Part 0 is missing labels for {len(missing)} source questions; finish the probes first")
    qs = [q for q in qs if labels[f"{q['dataset']}:{q['source_id']}"]]
if not qs:
    raise ValueError("no eligible questions")

def has_distractor(q):
    gold = {g["index"] for g in q["gold"] if g["worker"] == 0}
    return any(i not in gold for i in range(len(q["workers"][0])))

ARMS = {f"mask_a_{c}": (arms.masked_recompute, {
            "condition": c, "distractor_seed": DISTRACTOR_SEED, "thinking": "off",
            "max_answer_tokens": MAX_ANSWER_TOKENS, "score_gold": True})
        for c in ("blocked", "gold", "distractor", "except_gold", "full")}
if {"mask_a_blocked", "mask_a_full"}.intersection(SKIP):
    raise ValueError("The masking experiment requires both blocked and full endpoints; inspect the smoke failures")
ACTIVE_ARMS = {name: spec for name, spec in ARMS.items() if name not in SKIP}
eligible = {name: ([q for q in qs if has_distractor(q)] if opts["condition"] == "distractor" else qs)
            for name, (_, opts) in ACTIVE_ARMS.items()}
plan = []
for name, (arm, options) in ACTIVE_ARMS.items():
    path = results_path(MODEL, DTYPE, name)
    saved = results._read(path)
    results._check(path, saved, {"arm": arm.__name__, "options": options})
    present = sum(q["id"] in saved for q in eligible[name])
    plan.append({"arm": name, "eligible": len(eligible[name]), "already_saved": present,
                 "remaining": len(eligible[name]) - present})
display(pd.DataFrame(plan))
manifest = {"model": MODEL, "dtype": DTYPE, "commit": results._commit(), "smoke": SMOKE,
            "seeds": seeds, "n_per_cell": N_PER_CELL, "questions": len(qs), "skipped": SKIP,
            "composition_labels": None if SMOKE else labels_path.name,
            "arms": {name: options for name, (_, options) in ACTIVE_ARMS.items()}, "plan": plan}
manifest_name = "h1_masking-smoke-plan.json" if SMOKE else "h1_masking-plan.json"
manifest_path = results_path(MODEL, DTYPE, "x").parent / manifest_name
manifest_path.parent.mkdir(parents=True, exist_ok=True)
manifest_path.write_text(json.dumps(manifest, indent=2) + "\n")
print(len(qs), "eligible question rows;", sum(not has_distractor(q) for q in qs),
      "have no A distractor and cannot run that control")

In [ ]:
if any(item["remaining"] for item in plan):
    model = Model.load(MODEL, dtype=DTYPE)
    for name, (arm, options) in ACTIVE_ARMS.items():
        run(model, eligible[name], name, arm, **options)
else:
    print("All requested masking rows are already saved; no model load or inference needed.")

In [ ]:
df = load_results(qs, ACTIVE_ARMS, MODEL, DTYPE)
display(analysis.audit(df))
if SMOKE:
    analysis.peek(df)
counts = df.groupby("arm").id.nunique()
incomplete = {name: len(eligible[name]) - int(counts.get(name, 0)) for name in ACTIVE_ARMS
              if int(counts.get(name, 0)) != len(eligible[name])}
if incomplete:
    raise RuntimeError(f"Masking results are incomplete: {incomplete}")

## Check endpoints against saved H1 baselines

If B1 and B1-mask results are present (including through the Colab archive loader), use those rows directly. This does not rerun the baselines. Compare only paired questions and inspect discrepancies; floating-point differences can change a greedy answer. The masked endpoints are still recomputed to measure their gold-answer log-probabilities.

In [ ]:
import pandas as pd
BASELINES = {"b1": (arms.b1, {}), "b1_mask": (arms.b1_mask, {})}
baseline_df = load_results(qs, BASELINES, MODEL, DTYPE)
if len(baseline_df):
    endpoints = pd.concat([df, baseline_df], ignore_index=True)
    for masked, baseline in (("mask_a_full", "b1"), ("mask_a_blocked", "b1_mask")):
        print(masked, "−", baseline, "(paired EM points)")
        display(analysis.paired(endpoints, masked, baseline, by=("dataset", "placement")))
else:
    print("No saved B1/B1-mask rows found; endpoint comparisons are unavailable.")

## Paired comparisons

Means and intervals are paired by question and bootstrapped over source questions. Report the eligible count for each pair and inspect cut rates before interpreting EM. The log-probability outcomes use only rows that saved a gold score.

In [ ]:
display(analysis.accuracy(df, by=("dataset", "placement")))
for metric in ("em", "gold_logprob", "gold_mean_logprob"):
    for a, b in (("mask_a_full", "mask_a_blocked"), ("mask_a_gold", "mask_a_blocked"),
                 ("mask_a_distractor", "mask_a_blocked"), ("mask_a_gold", "mask_a_distractor"),
                 ("mask_a_except_gold", "mask_a_blocked")):
        if a not in ACTIVE_ARMS or b not in ACTIVE_ARMS:
            continue
        print(metric, a, "−", b)
        display(analysis.paired(df, a, b, by=("dataset", "placement"), metric=metric))

## Findings

_(Fill in after the full run; separate AB, BA, AA and BB.)_